# SupeRR Selector Season 2 — Consolidated Pipeline (Phases 1–3)
## Data Audit, Cricket State Reconstruction & Pressure Benchmark Modelling

**Competition:** Rajasthan Royals — SupeRR Selector Season 2  
**Scope:** Historical IPL & Men's T20I (2022–2026)  
**Boundary:** Consolidated execution of **Phase 1, Phase 2, and Phase 3 only**. (No Phase 4+ player ranking or scoring).

---

### Phase Alignment Summary:
- **Phase 0 (Methodology Design):** Closed. Established pre-delivery pressure principles, format separation (IPL vs T20I), par score benchmarking for Innings 1, and chase-demand strain for Innings 2.
- **Phase 1 (Data Understanding & Audit):** Ingests and audits 6 source CSVs, maps entity relationships, audits player identity, and establishes the approved primary population of 813 completed matches.
- **Phase 2 (Data Cleaning & Reconstruction):** Filters 192,235 regulation deliveries, classifies legal balls, resolves runs and bowler attribution, and reconstructs cumulative innings progression.
- **Phase 3 (Pressure Modelling & Benchmarks):** Reconstructs first-innings par benchmarks, second-innings chase targets and RRR benchmarks with adaptive fallback, formulates continuous pressure index $\in [0, 1]$, and exports pressure-labelled analytical datasets.


## 1. Environment Setup & Dependency Installation


In [1]:
# Install required libraries
%pip install -q pandas numpy ipython matplotlib seaborn scipy

import os
os.environ['MPLCONFIGDIR'] = '/tmp/matplotlib'
import sys
import json
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Explicit import of display and HTML for table and visual rendering
try:
    from IPython.display import display, HTML
except ImportError:
    display = print
    HTML = str

warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')

print('Environment initialized successfully.')
print(f'Pandas {pd.__version__}, NumPy {np.__version__}')


Note: you may need to restart the kernel to use updated packages.
Environment initialized successfully.
Pandas 3.1.1, NumPy 2.5.4


## 2. Phase 1 — Data Ingestion, Schema Audit & Population Definition

In Phase 1, we audit the six raw datasets provided in the Data Pack:
1. `ball_by_ball.csv`: 200,999 delivery records across 867 matches.
2. `matches.csv`: 883 match records with match metadata, teams, and outcomes.
3. `competitions.csv`: 141 competitions (IPL and bilateral/ICC T20Is).
4. `playing_eleven.csv`: Squad registrations and playing XIs per match.
5. `teams.csv`: Team metadata per competition.
6. `competition_team_player_mapping.csv`: Player-team-competition master mapping.

### Player Identity Disambiguation & Dropping Inactive/Phantom IDs:
Across 1,217 unique player IDs and 1,210 names in the raw mapping, 7 names were associated with multiple IDs:
- **0-Delivery Phantom IDs Dropped:** IDs that only exist as unplayed squad roster rows (Mohammad Ali ID 3083, Bashir Ahmad ID 39479, Rashid Khan ID 20172, Muhammad Farooq ID 28978, Basil Hameed ID 39444) have **0 deliveries in eligible matches** and are completely excluded from the delivery pipeline.
- **Negligible High-Pressure Volume:** Active IDs for Mohammad Ali (ID 20229: 1 HP ball) and Bashir Ahmad (ID 29833: 6 HP balls) have virtually zero high-pressure involvement and naturally fall below any meaningful sample threshold.
- **True Cross-National Namesakes Retained:** Only two same-name pairs represent active, distinct international players:
  - **Abdul Samad:** ID 7446 (India / SRH & LSG) vs ID 6922 (Pakistan / Pakistan A).
  - **Haider Ali:** ID 7338 (Pakistan top-order batter) vs ID 28756 (UAE spinner/all-rounder).
  They are preserved as separate entities using `player_id`.

### Approved Match Eligibility Criteria (from Phase 1 & 2 Reports):
- **Exclude 40 Abandoned / No Result matches:** Matches with no deliveries or interrupted before a competitive outcome are excluded from the primary completed-match population.
- **Exclude 30 Developmental T20 matches:** Mens T20 Emerging Teams Asia Cup and Asia Cup Rising Stars are developmental A-team matches outside the approved IPL/T20I scope.
- **Eligible Match Population:** Exactly **813 matches** (362 IPL, 451 T20I).



In [2]:
DATA_DIR = 'SupeRR Selector Season 2 | Data Pack'

ball_by_ball = pd.read_csv(f'{DATA_DIR}/ball_by_ball.csv')
matches = pd.read_csv(f'{DATA_DIR}/matches.csv')
competitions = pd.read_csv(f'{DATA_DIR}/competitions.csv')
playing_eleven = pd.read_csv(f'{DATA_DIR}/playing_eleven.csv')
teams = pd.read_csv(f'{DATA_DIR}/teams.csv')
player_mapping = pd.read_csv(f'{DATA_DIR}/competition_team_player_mapping.csv')

# Dataset Inventory Table
inventory = pd.DataFrame([
    {"Dataset": "ball_by_ball", "Rows": len(ball_by_ball), "Cols": len(ball_by_ball.columns), "Description": "Delivery-level ball records"},
    {"Dataset": "matches", "Rows": len(matches), "Cols": len(matches.columns), "Description": "Match metadata and official results"},
    {"Dataset": "competitions", "Rows": len(competitions), "Cols": len(competitions.columns), "Description": "Competition details and format"},
    {"Dataset": "playing_eleven", "Rows": len(playing_eleven), "Cols": len(playing_eleven.columns), "Description": "Squad / XI match registrations"},
    {"Dataset": "teams", "Rows": len(teams), "Cols": len(teams.columns), "Description": "Team identity per competition"},
    {"Dataset": "player_mapping", "Rows": len(player_mapping), "Cols": len(player_mapping.columns), "Description": "Player-team-competition registry"},
])

print("--- DATASET INVENTORY ---")
display(inventory)

# Identity Audit: Disambiguate the 7 Same-Name Cases and Classify Status
conflict_names = [
    "Abdul Samad", "Haider Ali", "Rashid Khan",
    "Mohammad Ali", "Bashir Ahmad", "Basil Hameed", "Muhammad Farooq"
]

conflict_audit = []
for name in conflict_names:
    sub = player_mapping[player_mapping['player_name'] == name]
    for pid in sub['player_id'].unique():
        p_maps = sub[sub['player_id'] == pid]
        t_names = list(p_maps['team_name'].unique())
        balls_bat = len(ball_by_ball[ball_by_ball['striker_id'] == pid])
        balls_bowl = len(ball_by_ball[ball_by_ball['bowler_id'] == pid])
        
        # Determine analytical status
        if balls_bat + balls_bowl == 0:
            status = "Dropped (0 Deliveries in Data)"
        elif name in ["Mohammad Ali", "Bashir Ahmad"] and balls_bat + balls_bowl < 50:
            status = "Negligible Volume (<50 balls total)"
        elif name in ["Abdul Samad", "Haider Ali"]:
            status = "Active Cross-National Namesake (Retained via player_id)"
        elif name in ["Basil Hameed", "Muhammad Farooq"]:
            status = "Consolidated under Senior UAE ID"
        else:
            status = "Active Player (Retained)"
            
        conflict_audit.append({
            "Player Name": name,
            "Player ID": pid,
            "Team(s)": ", ".join(t_names),
            "Deliveries Bowled/Faced": balls_bat + balls_bowl,
            "Pipeline Treatment": status
        })

print("\n--- PLAYER IDENTITY AUDIT (7 SAME-NAME CASES & DISPOSITION) ---")
display(pd.DataFrame(conflict_audit))



--- DATASET INVENTORY ---


,Dataset,Rows,Cols,Description
0,ball_by_ball,200999,33,Delivery-level ball records
1,matches,883,22,Match metadata and official results
2,competitions,141,4,Competition details and format
3,playing_eleven,28381,9,Squad / XI match registrations
4,teams,353,3,Team identity per competition
5,player_mapping,6413,6,Player-team-competition registry


In [3]:
# Match Population Filtering
mc = matches.merge(competitions, on='comp_id', how='left')

# Competition classification: IPL vs T20I vs Other_T20
def classify_comp(row):
    if row['match_format'] == 'T20I':
        return 'T20I'
    elif 'IPL' in str(row['comp_name']):
        return 'IPL'
    else:
        return 'Other_T20'

mc['comp_type'] = mc.apply(classify_comp, axis=1)

match_ids_with_balls = set(ball_by_ball['match_id'].unique())

# Retain only completed matches in target competitions with delivery records
eligible = mc[
    (~mc['result'].isin(['No Result', 'Match Abandoned'])) &
    (mc['match_id'].isin(match_ids_with_balls)) &
    (mc['comp_type'].isin(['IPL', 'T20I']))
].copy()

eligible_match_ids = set(eligible['match_id'])

# Verification Summary
pop_summary = pd.DataFrame([
    {"Population Metric": "Total Matches in matches.csv", "Count": len(mc)},
    {"Population Metric": "Abandoned / No Result Matches Excluded", "Count": mc['result'].isin(['No Result', 'Match Abandoned']).sum()},
    {"Population Metric": "Non-IPL T20 Matches Excluded (Emerging/Rising)", "Count": (mc['comp_type'] == 'Other_T20').sum()},
    {"Population Metric": "Matches with Zero Deliveries Excluded", "Count": (~mc['match_id'].isin(match_ids_with_balls)).sum()},
    {"Population Metric": "Approved Eligible Match Population", "Count": len(eligible)},
    {"Population Metric": "  — IPL Completed Matches", "Count": (eligible['comp_type'] == 'IPL').sum()},
    {"Population Metric": "  — T20I Completed Matches", "Count": (eligible['comp_type'] == 'T20I').sum()},
])

print("--- MATCH POPULATION AUDIT ---")
display(pop_summary)

print("\nEligible Matches by Season & Format:")
display(eligible.groupby(['season', 'comp_type']).size().unstack(fill_value=0))


--- MATCH POPULATION AUDIT ---


,Population Metric,Count
0,Total Matches in matches.csv,883
1,Abandoned / No Result Matches Excluded,40
2,Non-IPL T20 Matches Excluded (Emerging/Rising),30
3,Matches with Zero Deliveries Excluded,16
4,Approved Eligible Match Population,813
5,— IPL Completed Matches,362
6,— T20I Completed Matches,451



Eligible Matches by Season & Format:


comp_type,IPL,T20I
season,,
2022,74,91
2022/23,0,12
2023,73,61
2023/24,0,3
2024,71,101
2024/25,0,3
2025,71,112
2026,73,68


## 3. Phase 2 — Data Cleaning & Cricket State Reconstruction

In Phase 2, we reconstruct the delivery-level match progression across the approved 813 matches:
1. **Regulation Deliveries Only:** Innings 1 and Innings 2 deliveries are retained (192,235 balls). Super Overs (innings 3 & 4: 38 deliveries across 6 matches) are excluded from regulation analysis.
2. **Legal Ball Evidence Classification:**
   - Deliveries with `extras_type` in `['wide', 'Overstep']` are flagged as **non-legal balls** (7,976 deliveries).
   - Normal deliveries, byes, legbyes, and rare unresolved records are retained as **legal balls** (184,259 deliveries).
3. **Runs Attribution:**
   - `runs`: Batter runs off the bat.
   - `total_delivery_runs`: Team score contribution (batter runs + 1 penalty for wide/no-ball/bye/legbye).
   - `bowler_runs_conceded`: Runs charged to the bowler (batter runs + wide/no-ball; byes and legbyes explicitly excluded).
   - `is_dot_ball`: Batter scored 0 off a legal delivery.
4. **Pre-Delivery State Reconstruction:**
   - Pre-delivery cumulative score and pre-delivery wickets fallen ($0 \le W \le 9$) are strictly computed **prior to delivery outcome**.
   - Innings stage classified as: **Powerplay (Overs 1–6)**, **Middle (Overs 7–15)**, **Death (Overs 16–20)**.


In [4]:
# Filter to eligible matches, regulation innings only
df = ball_by_ball[
    (ball_by_ball['match_id'].isin(eligible_match_ids)) &
    (ball_by_ball['innings'].isin([1, 2]))
].copy()

# Legal ball classification
df['is_legal_ball'] = ~df['extras_type'].isin(['wide', 'Overstep'])

# Runs attribution
df['total_delivery_runs'] = df['runs'].copy()
df.loc[df['extras_type'] == 'wide', 'total_delivery_runs'] = 1
df.loc[df['extras_type'] == 'Overstep', 'total_delivery_runs'] = df.loc[df['extras_type'] == 'Overstep', 'runs'] + 1
df.loc[df['extras_type'] == 'legbyes', 'total_delivery_runs'] = 1
df.loc[df['extras_type'] == 'byes', 'total_delivery_runs'] = 1

# Bowler conceded runs (byes/legbyes NOT charged to bowler)
df['bowler_runs_conceded'] = df['runs'].copy()
df.loc[df['extras_type'] == 'wide', 'bowler_runs_conceded'] = 1
df.loc[df['extras_type'] == 'Overstep', 'bowler_runs_conceded'] = df.loc[df['extras_type'] == 'Overstep', 'runs'] + 1
df.loc[df['extras_type'].isin(['byes', 'legbyes']), 'bowler_runs_conceded'] = 0

# Dot ball flag
df['is_dot_ball'] = (df['runs'] == 0) & df['is_legal_ball'] & (~df['extras_type'].isin(['wide', 'Overstep']))

# Attach match context metadata
match_cols = ['match_id', 'comp_id', 'comp_type', 'comp_name', 'match_format',
              'season', 'match_date', 'venue_country', 'toss_opted',
              'win_team_id', 'result', 'revised_target',
              'first_inn_revised_overs', 'second_inn_revised_overs', 'knock_out']
df = df.merge(eligible[match_cols], on='match_id', how='left')

# Sort chronologically by match, innings, and delivery sequence
df = df.sort_values(['match_id', 'innings', 'innings_ball_num']).reset_index(drop=True)

# Reconstruct pre-delivery cumulative score
df['cumulative_runs'] = df.groupby(['match_id', 'innings'])['total_delivery_runs'].cumsum()
df['pre_delivery_score'] = df['cumulative_runs'] - df['total_delivery_runs']

# Reconstruct pre-delivery wickets fallen
df['wicket_event'] = df['is_wicket'].astype(int)
df['cumulative_wickets'] = df.groupby(['match_id', 'innings'])['wicket_event'].cumsum()
df['pre_delivery_wickets'] = df['cumulative_wickets'] - df['wicket_event']
df['wickets_remaining'] = 10 - df['pre_delivery_wickets']

# Over progression and innings stage
df['over_number'] = df['overid']

def get_stage(over):
    if over <= 6: return 'Powerplay'
    elif over <= 15: return 'Middle'
    else: return 'Death'

df['innings_stage'] = df['over_number'].apply(get_stage)

# Legal ball progression and estimated remaining balls
df['legal_ball_number'] = df.groupby(['match_id', 'innings'])['is_legal_ball'].cumsum()
df['legal_balls_remaining'] = (120 - df['legal_ball_number']).clip(lower=0)

print(f"Regulation Deliveries Processed: {len(df):,}")
print(f"  Legal Balls: {df['is_legal_ball'].sum():,}")
print(f"  Wides & No-Balls: {(~df['is_legal_ball']).sum():,}")
display(df[['match_id', 'innings', 'over_number', 'innings_stage', 'pre_delivery_score', 'pre_delivery_wickets', 'legal_balls_remaining']].head())


Regulation Deliveries Processed: 192,235
  Legal Balls: 184,259
  Wides & No-Balls: 7,976


,match_id,innings,over_number,innings_stage,pre_delivery_score,pre_delivery_wickets,legal_balls_remaining
0,13511,1,1,Powerplay,0,0,119
1,13511,1,1,Powerplay,0,0,118
2,13511,1,1,Powerplay,0,0,117
3,13511,1,1,Powerplay,0,0,116
4,13511,1,1,Powerplay,0,0,115


## 4. Phase 3 — Pressure Modelling & Benchmark Construction

### 4.1 First-Innings Pressure Formulation (Par Score Deviation)
In the first innings, there is no chase target. Pressure is driven by:
1. **Wicket Depletion:** $	ext{Wicket Pressure} = rac{	ext{Pre-Delivery Wickets}}{9} \in [0, 1]$
2. **Par Score Strain:** Deviation of current cumulative score from historical par for that exact over and wicket state:
   $$	ext{Par Score Benchmark} = \mu(	ext{Score} \mid 	ext{Format}, 	ext{Over}, 	ext{Wickets})$$
   $$	ext{Standardized Deviation} = rac{	ext{Actual Score} - 	ext{Par Score}}{\sigma(	ext{Par Score})}$$
   Scoring below par increases pressure. Scaled to $[0, 1]$ by clamping $\pm 3\sigma$.
3. **Stage Urgency:** Death overs carry higher stakes than powerplay: $	ext{Stage Pressure} = rac{	ext{Over} - 1}{19}$

$$	ext{Pressure}_{	ext{Inn1}} = 0.35 \cdot 	ext{Wicket Pressure} + 0.40 \cdot 	ext{Par Pressure} + 0.25 \cdot 	ext{Stage Pressure}$$

### 4.2 Second-Innings Pressure Formulation (Relative Chase Demand)
In the second innings, pressure is determined by chase demand:
1. **Chase Target:** First Innings Total + 1 (with official DLS `revised_target` integrated for rain-affected games).
2. **Required Run Rate (RRR):** $	ext{RRR} = rac{	ext{Runs Required}}{	ext{Balls Remaining} / 6}$
3. **Excess Demand Strain:** Difference between RRR and historical expected scoring rate for that over and wicket state:
   $$\Delta_{	ext{Demand}} = 	ext{RRR} - 	ext{Expected Run Rate}(	ext{Format}, 	ext{Over}, 	ext{Wickets})$$

$$	ext{Pressure}_{	ext{Inn2}} = 0.30 \cdot 	ext{Wicket Pressure} + 0.45 \cdot 	ext{Chase Demand Pressure} + 0.25 \cdot 	ext{Stage Pressure}$$

### 4.3 Adaptive Benchmark Fallback
To ensure complete coverage without cross-format contamination:
- Exact state matching: `(comp_type, over_number, pre_delivery_wickets)`.
- Fallback for sparse/unobserved states: Broadens to stage-level format average `(comp_type, innings_stage)`.


In [5]:
# 1. First Innings Totals & Chase Targets
first_innings_totals = (
    df[df['innings'] == 1]
    .groupby('match_id')
    .agg(
        first_inn_total=('total_delivery_runs', 'sum'),
        first_inn_wickets=('is_wicket', 'sum'),
        first_inn_legal_balls=('is_legal_ball', 'sum')
    )
    .reset_index()
)

first_innings_totals = first_innings_totals.merge(
    eligible[['match_id', 'revised_target']].drop_duplicates(), on='match_id', how='left'
)
first_innings_totals['chase_target'] = first_innings_totals['first_inn_total'] + 1
mask_dls = first_innings_totals['revised_target'].notna()
first_innings_totals.loc[mask_dls, 'chase_target'] = first_innings_totals.loc[mask_dls, 'revised_target']

# Merge chase target to deliveries
df = df.merge(first_innings_totals[['match_id', 'first_inn_total', 'chase_target']], on='match_id', how='left')

# Second innings runs required and RRR
second_mask = df['innings'] == 2
df.loc[second_mask, 'runs_required'] = df.loc[second_mask, 'chase_target'] - df.loc[second_mask, 'pre_delivery_score']
df.loc[second_mask, 'required_run_rate'] = np.where(
    df.loc[second_mask, 'legal_balls_remaining'] > 0,
    df.loc[second_mask, 'runs_required'] / (df.loc[second_mask, 'legal_balls_remaining'] / 6),
    np.where(df.loc[second_mask, 'runs_required'] > 0, 36.0, 0.0)
)

print(f"First Innings Aggregated: {len(first_innings_totals)} matches")
print(f"Mean 1st Innings Total: {first_innings_totals['first_inn_total'].mean():.1f} runs")
print(f"DLS Revised Targets Applied: {mask_dls.sum()} matches")


First Innings Aggregated: 813 matches
Mean 1st Innings Total: 174.9 runs
DLS Revised Targets Applied: 22 matches


In [6]:
# 2. Innings 1 Par Score Benchmarks with Adaptive Fallback
fi_df = df[df['innings'] == 1]

# Exact state benchmark
par_benchmarks = (
    fi_df.groupby(['comp_type', 'over_number', 'pre_delivery_wickets'])
    .agg(
        par_score_mean=('pre_delivery_score', 'mean'),
        par_score_std=('pre_delivery_score', 'std'),
        par_delivery_count=('pre_delivery_score', 'count')
    )
    .reset_index()
)
par_benchmarks['par_score_std'] = par_benchmarks['par_score_std'].fillna(0)

# Stage-level fallback for rare/sparse wicket states
par_stage_fallback = (
    fi_df.groupby(['comp_type', 'innings_stage'])
    .agg(
        fallback_par_score=('pre_delivery_score', 'mean'),
        fallback_par_std=('pre_delivery_score', 'std')
    )
    .reset_index()
)

# Merge benchmarks
df = df.merge(par_benchmarks, on=['comp_type', 'over_number', 'pre_delivery_wickets'], how='left')
df = df.merge(par_stage_fallback, on=['comp_type', 'innings_stage'], how='left')

# Calculate Innings 1 scoring deviation with fallback
first_mask = df['innings'] == 1
effective_par_mean = df.loc[first_mask, 'par_score_mean'].fillna(df.loc[first_mask, 'fallback_par_score'])
effective_par_std = df.loc[first_mask, 'par_score_std'].fillna(df.loc[first_mask, 'fallback_par_std']).clip(lower=1.0)

df.loc[first_mask, 'score_vs_par'] = df.loc[first_mask, 'pre_delivery_score'] - effective_par_mean
df.loc[first_mask, 'score_vs_par_std'] = df.loc[first_mask, 'score_vs_par'] / effective_par_std

print(f"First-Innings Benchmark States: {len(par_benchmarks)}")
print(f"Median Observations per State: {par_benchmarks['par_delivery_count'].median():.0f}")


First-Innings Benchmark States: 320
Median Observations per State: 219


In [7]:
# 3. Innings 2 Chase Rate Benchmarks with Adaptive Fallback
si_df = df[df['innings'] == 2]

# Exact state benchmark: expected scoring rate per ball
chase_benchmarks = (
    si_df.groupby(['comp_type', 'over_number', 'pre_delivery_wickets'])
    .agg(
        expected_scoring_rate=('total_delivery_runs', 'mean'),
        expected_sr_std=('total_delivery_runs', 'std'),
        chase_delivery_count=('total_delivery_runs', 'count')
    )
    .reset_index()
)
chase_benchmarks['expected_run_rate'] = chase_benchmarks['expected_scoring_rate'] * 6

# Stage-level fallback
chase_stage_fallback = (
    si_df.groupby(['comp_type', 'innings_stage'])
    .agg(fallback_expected_run_rate=('total_delivery_runs', lambda s: s.mean() * 6))
    .reset_index()
)

# Merge chase benchmarks
df = df.merge(chase_benchmarks[['comp_type', 'over_number', 'pre_delivery_wickets', 'expected_run_rate', 'chase_delivery_count']],
              on=['comp_type', 'over_number', 'pre_delivery_wickets'], how='left')
df = df.merge(chase_stage_fallback, on=['comp_type', 'innings_stage'], how='left')

# Calculate Relative Chase Demand
effective_exp_rr = df.loc[second_mask, 'expected_run_rate'].fillna(df.loc[second_mask, 'fallback_expected_run_rate'])
df.loc[second_mask, 'chase_pressure_diff'] = df.loc[second_mask, 'required_run_rate'] - effective_exp_rr

print(f"Second-Innings Benchmark States: {len(chase_benchmarks)}")
print(f"Median Observations per State: {chase_benchmarks['chase_delivery_count'].median():.0f}")


Second-Innings Benchmark States: 327
Median Observations per State: 201


In [8]:
# 4. Pressure Index Construction & Classification
# First innings continuous pressure
fi = df[df['innings'] == 1].index
df.loc[fi, 'wicket_pressure'] = df.loc[fi, 'pre_delivery_wickets'] / 9.0
svp = df.loc[fi, 'score_vs_par_std'].clip(-3, 3)
df.loc[fi, 'par_pressure'] = (-svp + 3) / 6  # -3 -> 1.0 (high pressure), +3 -> 0.0 (low)
df.loc[fi, 'stage_pressure'] = (df.loc[fi, 'over_number'] - 1) / 19.0

df.loc[fi, 'pressure_index'] = (
    0.35 * df.loc[fi, 'wicket_pressure'] +
    0.40 * df.loc[fi, 'par_pressure'] +
    0.25 * df.loc[fi, 'stage_pressure']
)

# Second innings continuous pressure
si = df[df['innings'] == 2].index
df.loc[si, 'wicket_pressure'] = df.loc[si, 'pre_delivery_wickets'] / 9.0
cpd = df.loc[si, 'chase_pressure_diff'].fillna(0).clip(-6, 12)
df.loc[si, 'chase_demand_pressure'] = (cpd + 6) / 18  # -6 -> 0.0, +12 -> 1.0
df.loc[si, 'stage_pressure'] = (df.loc[si, 'over_number'] - 1) / 19.0

df.loc[si, 'pressure_index'] = (
    0.30 * df.loc[si, 'wicket_pressure'] +
    0.45 * df.loc[si, 'chase_demand_pressure'] +
    0.25 * df.loc[si, 'stage_pressure']
)

# Categorization into High (>=0.60), Medium (0.35-0.60), Low (<0.35)
def categorize_pressure(p):
    if pd.isna(p): return 'Unknown'
    if p >= 0.60: return 'High'
    elif p >= 0.35: return 'Medium'
    else: return 'Low'

df['pressure_category'] = df['pressure_index'].apply(categorize_pressure)

print("--- PRESSURE INDEX DISTRIBUTION ---")
print(f"Mean Pressure Index: {df['pressure_index'].mean():.3f}")
print(f"Median Pressure Index: {df['pressure_index'].median():.3f}")
print(f"Zero / Null Pressure Count: {df['pressure_index'].isna().sum()}")

cat_df = df['pressure_category'].value_counts().to_frame('Delivery Count')
cat_df['Percentage'] = (cat_df['Delivery Count'] / len(df) * 100).round(2)
display(cat_df)

print("\nPressure by Innings Stage:")
display(df.groupby('innings_stage')['pressure_category'].value_counts().unstack(fill_value=0))


--- PRESSURE INDEX DISTRIBUTION ---
Mean Pressure Index: 0.417
Median Pressure Index: 0.390
Zero / Null Pressure Count: 0


,Delivery Count,Percentage
pressure_category,,
Low,80458,41.850
Medium,79250,41.230
High,32527,16.920



Pressure by Innings Stage:


pressure_category,High,Low,Medium
innings_stage,,,
Death,24133,1634,16910
Middle,8376,24689,55538
Powerplay,18,54135,6802


## 5. Phase 3 Validation Checks & Deliverable Exports

In accordance with Phase 3 of the project roadmap, we:
1. Run formal consistency checks (null checks, bounded values, innings completeness).
2. Export the Phase 3 analytical deliverables:
   - `deliveries_pressure_labelled.csv`: Master delivery table with continuous pressure scores and categorical pressure tags.
   - `eligible_matches.csv`: Metadata for the 813 eligible matches.
   - `par_score_benchmarks.csv`: First-innings par score benchmark table.
   - `chase_rate_benchmarks.csv`: Second-innings expected scoring rate benchmark table.
   - `first_innings_totals.csv`: First-innings totals and target score crosswalk.


In [9]:
# Formal Validation Checks
checks = []
checks.append(("Zero null pressure index values", df['pressure_index'].isna().sum() == 0))
checks.append(("Pressure index strictly bounded in [0, 1]", df['pressure_index'].between(0, 1).all()))
checks.append(("Exact eligible match population (813 matches)", df['match_id'].nunique() == 813))
checks.append(("Exact regulation delivery population (192,235 balls)", len(df) == 192235))
checks.append(("Only IPL and T20I formats present", set(df['comp_type'].unique()) == {'IPL', 'T20I'}))
checks.append(("High Pressure represents crunch situations (~15-20%)", 15 <= (df['pressure_category'] == 'High').mean() * 100 <= 20))

print("--- VALIDATION RESULTS ---")
for name, passed in checks:
    status = "✅ PASSED" if passed else "❌ FAILED"
    print(f"{status:10s} : {name}")

assert all(p for _, p in checks), "Validation failed!"


--- VALIDATION RESULTS ---
✅ PASSED   : Zero null pressure index values
✅ PASSED   : Pressure index strictly bounded in [0, 1]
✅ PASSED   : Exact eligible match population (813 matches)
✅ PASSED   : Exact regulation delivery population (192,235 balls)
✅ PASSED   : Only IPL and T20I formats present
✅ PASSED   : High Pressure represents crunch situations (~15-20%)


In [10]:
# Export Deliverables (Phase 3 Complete)
export_cols = [
    'match_id', 'innings', 'overid', 'over_ball_num', 'innings_ball_num',
    'striker_id', 'striker_name', 'non_striker_id', 'non_striker_name',
    'bowler_id', 'bowler_name', 'batting_team_id', 'bowling_team_id',
    'batting_type', 'bowling_type', 'pace_or_spin', 'batting_order',
    'runs', 'total_delivery_runs', 'bowler_runs_conceded',
    'is_four', 'is_six', 'is_wicket', 'dismissal_type', 'dismissed_player_id',
    'extras_type', 'is_legal_ball', 'is_dot_ball',
    'comp_type', 'comp_name', 'season', 'match_date', 'venue_country',
    'toss_opted', 'win_team_id', 'result', 'knock_out',
    'over_number', 'innings_stage',
    'pre_delivery_score', 'cumulative_runs',
    'pre_delivery_wickets', 'wickets_remaining',
    'legal_ball_number', 'legal_balls_remaining',
    'first_inn_total', 'chase_target', 'runs_required', 'required_run_rate',
    'par_score_mean', 'score_vs_par', 'score_vs_par_std',
    'expected_run_rate', 'chase_pressure_diff',
    'pressure_index', 'pressure_category',
    'wicket_pressure', 'stage_pressure', 'par_pressure', 'chase_demand_pressure'
]
export_cols = [c for c in export_cols if c in df.columns]

df_export = df[export_cols].copy()
df_export.to_csv('deliveries_pressure_labelled.csv', index=False)
eligible.to_csv('eligible_matches.csv', index=False)
par_benchmarks.to_csv('par_score_benchmarks.csv', index=False)
chase_benchmarks.to_csv('chase_rate_benchmarks.csv', index=False)
first_innings_totals.to_csv('first_innings_totals.csv', index=False)

print("--- EXPORTED PHASE 3 DELIVERABLES ---")
for fname in ['deliveries_pressure_labelled.csv', 'eligible_matches.csv',
              'par_score_benchmarks.csv', 'chase_rate_benchmarks.csv',
              'first_innings_totals.csv']:
    sz = os.path.getsize(fname) / (1024 * 1024)
    print(f"  {fname:35s} : {sz:6.2f} MB")

print(f"\nPhase 3 Deliverables generated successfully.")
print(f"Master deliveries dataset: {len(df_export):,} rows × {len(export_cols)} columns.")


--- EXPORTED PHASE 3 DELIVERABLES ---
  deliveries_pressure_labelled.csv    :  80.62 MB
  eligible_matches.csv                :   0.15 MB
  par_score_benchmarks.csv            :   0.01 MB
  chase_rate_benchmarks.csv           :   0.02 MB
  first_innings_totals.csv            :   0.02 MB

Phase 3 Deliverables generated successfully.
Master deliveries dataset: 192,235 rows × 60 columns.


## 6. Phase 3 Formal Closure & Next Steps

### Deliverable Verification:
- ✅ **Reconstructed pre-delivery states:** Completed for both first and second innings.
- ✅ **Expected scoring benchmarks:** 320 first-innings par states and 327 second-innings chase states calculated.
- ✅ **Adaptive fallback rules:** Structured stage-level fallback implemented without cross-format pooling.
- ✅ **Standardized pressure indicators:** Continuous `pressure_index` $\in [0, 1]$ and validated categories.
- ✅ **Data deliverables exported:** `deliveries_pressure_labelled.csv` and associated benchmark files verified on disk.

### Boundary Check:
**Phase 3 is complete and closed.**  
As per project roadmap guidelines, Phase 4 (Player Metrics & Scoring Methodology) and subsequent phases remain strictly separate and will proceed only upon authorization.
